In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Task 2: Write your code here:
 #1. Read the dataset
# Build the full CSV path
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)
df.head()


In [ ]:

# Task 3: Write your code here:
df.info()





In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(8, 5))
sns.histplot(df["Delivery_Time"], bins=30, kde=True)
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Delivery Time")
plt.show()


In [ ]:
# Task 1: Write your code here:

if "Order_ID" in df.columns:
    df = df.drop(columns=["Order_ID"])


In [ ]:
# Task 2: Write your code here:
# Check missing values df.isnull().sum()
df.isnull().sum()
# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = df.select_dtypes(include=["object"]).columns

# Fill numeric missing values with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical missing values with mode
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum()


In [ ]:
# Task 3: Write your code here:
# Check duplicates
print("Duplicates before:", df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

print("Duplicates after:", df.duplicated().sum())


In [ ]:
# Task 4: Write your code here:
# One Hot Encode categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df_encoded.head()


In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Identify the target column
target_col = "Delivery_Time"

# 2. Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# 3. One-Hot Encode categorical columns
categorical_cols = X.select_dtypes(include=["object"]).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# 4. Apply StandardScaler to numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]


In [ ]:
# Task 6: Write your code here:
# Plot to visually inspect imbalance
sns.histplot(y, kde=True)
plt.show()

# If the distribution looks continuous (which it does),
# the target is NOT imbalanced.


In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Features and target
X = X_scaled        # from previous scaling step
y = y               # Delivery_Time


In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mae_scores.append(mae)

print("MAE scores for each fold:", mae_scores)
print("Average MAE:", np.mean(mae_scores))


In [ ]:
# Train final model on scaled features
final_model = RandomForestRegressor(random_state=42)
final_model.fit(X_scaled, y)

# Use the original DataFrame to get feature names
feature_names = X.columns
importances = final_model.feature_importances_

plt.figure(figsize=(10,6))
sns.barplot(x=importances, y=feature_names)
plt.title("Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()


In [ ]:
# Task 2: Write your code here:
y_pred = final_model.predict(X_scaled)

plt.figure(figsize=(8,5))
sns.histplot(y_pred, kde=True, bins=30)
plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.show()


In [ ]:
!pip install catboost
from catboost import CatBoostRegressor

# Task Bonus: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Model 1: Random Forest
    rf = RandomForestRegressor(random_state=42)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    # Model 2: CatBoost
    cb = CatBoostRegressor(verbose=0, random_state=42)
    cb.fit(X_train, y_train)
    cb_pred = cb.predict(X_test)

    # Average predictions
    avg_pred = (rf_pred + cb_pred) / 2

    # MAE on averaged predictions
    mae = mean_absolute_error(y_test, avg_pred)
    mae_scores.append(mae)

# Print results
print("MAE scores for each fold:", mae_scores)
print("Average MAE:", np.mean(mae_scores))
